<a href="https://colab.research.google.com/github/spbromberg/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
duplicate_count = df.duplicated().sum()
# Count duplicate rows before removing them.

df = df.drop_duplicates().copy()
# Remove duplicate rows and make a fresh copy.

log(
    'Drop duplicates',
    'Removed exact duplicate transaction rows.',
    duplicate_count
)
# Record the decision in the cleaning log.


print('Rows remaining:', len(df))

[Drop duplicates] Removed exact duplicate transaction rows. (15 row(s))
Rows remaining: 300


### TODO 2 — clean `price` -> float

In [4]:
df['price'] = df['price'].str.replace('$', '', regex=False)
# Remove the dollar sign from price

df['price'] = df['price'].astype(float)
# Convert the cleaned price column to float type

log(
    'Clean price',
    'Removed dollar signs and converted price to float.',
    0
)
# Every row had a usable price, so no rows were dropped

print(df['price'].head())

[Clean price] Removed dollar signs and converted price to float. (0 row(s))
0    12.0
1     7.5
2     7.5
3     7.5
4     7.5
Name: price, dtype: float64


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
# Convert quantity to numeric values.

bad_qty = df['qty'].isna() | (df['qty'] < 0)
# Find rows with a missing or negative quantity

bad_qty_count = bad_qty.sum()
# Count those rows before removing them

df = df[~bad_qty].copy()
# Keep only rows with quantity of at least 1.

log(
    'Clean quantity',
    'Removed rows with missing or negative quantities.',
    bad_qty_count
)

print('Rows remaining:', len(df))

[Clean quantity] Removed rows with missing or negative quantities. (25 row(s))
Rows remaining: 275


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

# Map each inconsistent spelling to one standard product name
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

# Count rows whose item label will change
item_rows_changed = (df['item'] != df['item'].map(ITEM_MAP)).sum()

# Apply the standardized names
df['item'] = df['item'].map(ITEM_MAP)

log(
    'Canonicalize item',
    'Combined six item spellings into three standard product names.',
    item_rows_changed
)

print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[Canonicalize item] Combined six item spellings into three standard product names. (126 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear'
}
# Fix category spelling and capitalization
# Combined Apparel with Merch because both are similar items

category_rows_changed = (df['category'] != df['category'].map(CATEGORY_MAP)).sum()
# Count rows whose category label will change.

df['category'] = df['category'].map(CATEGORY_MAP)
# Apply the category mapping

log(
    'Normalize category',
    'Standardized category labels and combined Apparel into Merch.',
    category_rows_changed
)

print(df['category'].value_counts())

[Normalize category] Standardized category labels and combined Apparel into Merch. (132 row(s))
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item'].unique()) == {
    'Cheeseburger',
    'Foam Finger',
    'Rain Poncho'
    }
assert set(df['category'].unique()) == {
    'Food',
    'Merch',
    'RainGear'
}
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']
# revenue is price times quantity sold for each item

revenue_by_category = df.groupby('category')['revenue'].sum()
revenue_by_category = revenue_by_category.sort_values(ascending=False)
revenue_by_category = revenue_by_category.round(2)
# Add revenue within each category and sort from highest to lowest

print('Revenue by category:')
print(revenue_by_category)

overall_revenue = df['revenue'].sum().round(2)
# Calculate the overall revenue total

print()
print('Overall revenue:', overall_revenue)

Revenue by category:
category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64

Overall revenue: 4740.0


**What I would tell the vendor:** Stock more food, especially cheeseburgers, because Food generated the highest total revenue at $1,656.00.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,Drop duplicates,Removed exact duplicate transaction rows.,15
1,Clean price,Removed dollar signs and converted price to fl...,0
2,Clean quantity,Removed rows with missing or negative quantities.,25
3,Canonicalize item,Combined six item spellings into three standar...,126
4,Normalize category,Standardized category labels and combined Appa...,132


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

**a)** Dropping duplicates changed the revenue total the most. Revenue decreased from 4,852.50 dollars before duplicate removal to 4,594.50 dollars after duplicate removal (a decrease of 258 dollars). Removing rows with missing or negative quantities then changed revenue from 4,594.50 dollars to 4,740 dollars.

**b)** A reasonable person could have treated negative quantities as returns rather than deleting those rows. The negative quantities would remain in the dataset and reduce total revenue. I removed them because the lab describes the data as sales transactions, and a negative quantity is more likely to be a data-entry error than a return. Removing them produced a reported total revenue of 4,740 dollars.